In [ ]:
import openslide as ops
import os
from glob import glob
from tqdm import tqdm
import pyvips
from pathlib import Path

In [ ]:
Unstain_slide_path = Path('../../data/HnE_n_UNStaining/Un-Stained')
HnE_slide_path = Path('../../data/HnE_n_UNStaining/C_Stained')
Unstain_slide_tiff_path = Path('../../data/HnE_n_UNStaining/Un-Stained-tiff')
HnE_slide_tiff_path = Path('../../data/HnE_n_UNStaining/C_Stained-tiff')
SOURCE_MPP = 0.5
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)
create_folder(Unstain_slide_tiff_path)
create_folder(HnE_slide_tiff_path)
unstain_by_name = {path.name: path for path in Unstain_slide_path.glob('*.jpg')}
hne_by_name = {path.name: path for path in HnE_slide_path.glob('*.jpg')}
common_names = sorted(unstain_by_name.keys() & hne_by_name.keys())
Unstain_slide_list = [unstain_by_name[name] for name in common_names]
HnE_slide_list = [hne_by_name[name] for name in common_names]
print(f'Matched JPG pairs: {len(common_names)}')

In [ ]:
def convert_jpg2openslide_tiff(jpg_path, tiff_path, source_mpp=0.5):
    jpg_path = Path(jpg_path)
    tiff_path = Path(tiff_path)

    if not jpg_path.is_file():
        raise FileNotFoundError(jpg_path)

    tiff_path.parent.mkdir(parents=True, exist_ok=True)

    if source_mpp <= 0:
        raise ValueError(f'source_mpp must be positive: {source_mpp}')

    temporary_path = tiff_path.with_name(f'.{tiff_path.stem}.tmp.tiff')
    if temporary_path.exists():
        temporary_path.unlink()

    # pyvips의 xres/yres 단위는 pixels/mm이다.
    pixels_per_mm = 1000.0 / source_mpp
    image = pyvips.Image.new_from_file(str(jpg_path), access='sequential')
    image.tiffsave(
        str(temporary_path),
        tile=True,
        tile_width=256,
        tile_height=256,
        pyramid=True,
        compression="jpeg",
        Q=90,
        bigtiff=True,
        xres=pixels_per_mm,
        yres=pixels_per_mm,
        resunit='cm',
    )

    with ops.OpenSlide(str(temporary_path)) as slide:
        saved_mpp_x = float(slide.properties['openslide.mpp-x'])
        saved_mpp_y = float(slide.properties['openslide.mpp-y'])
    if abs(saved_mpp_x - source_mpp) > 1e-6 or abs(saved_mpp_y - source_mpp) > 1e-6:
        raise RuntimeError(
            f'MPP verification failed: expected={source_mpp}, saved=({saved_mpp_x}, {saved_mpp_y})'
        )
    temporary_path.replace(tiff_path)


for unstain_jpg, hne_jpg in tqdm(
    zip(Unstain_slide_list, HnE_slide_list), total=len(common_names)
):
    convert_jpg2openslide_tiff(
        unstain_jpg,
        Unstain_slide_tiff_path / unstain_jpg.with_suffix('.tiff').name,
        source_mpp=SOURCE_MPP,
    )
    convert_jpg2openslide_tiff(
        hne_jpg,
        HnE_slide_tiff_path / hne_jpg.with_suffix('.tiff').name,
        source_mpp=SOURCE_MPP,
    )
